source: https://www.bea.gov/data/gdp/gross-domestic-product

Creating Vintage Time Series of BEA estimations and revisions of US GDP.

In [1]:
import numpy as np
# import polars as pl
import pandas as pd
# import os
# import glob
import re
# from pprint import pprint
# import matplotlib.pyplot as plt
# from matplotlib.animation import FuncAnimation
import sys 

In [2]:
# Download current version of the excel file from
# https://www.bea.gov/sites/default/files/2026-02/gdp-gdi-vintage-history.xlsx
# read excel file from sheet 'Vintage History' into data frame 'raw'
raw  = pd.read_excel('gdp-gdi-vintage-history.xlsx', sheet_name='Vintage History',
    skiprows=5,
    header=None)

Table needs to be cleaned:
- each quarter with its' estimations and revisions is listed in an individual table
    - Identify boundries of the tables using column (0)
- Create meaningfull headers

In [ ]:
# For each Quarter there is an individual table
# Fill nan in first column with new column to create quarter date
# New column name
raw['Quarter'] = raw[0]
# Down fill 'Quarter'
raw['Quarter'] = raw['Quarter'].ffill()

In [8]:
# Transfer to Dataframe 
df = raw[[ 'Quarter', 1, 2, 3, 4, 5, 6]].copy()

In [11]:
# Assign column names
df.columns = [
    "Quarter",
    "Type",
    "GDP",
    "GDI",
    "GDP_%_Change",
    "GDI_%_Change",
    "Release_Date"
]

In [13]:
# Split Release_Date into Release_Date and Note 
df['Release_Date'] = df['Release_Date'].astype(str)
df['Note'] = df['Release_Date'].str[12:].str.strip().replace("",pd.NA)
df['Release_Date'] = df['Release_Date'].str[:12]

In [15]:
# Change Release_Date into Date time
df['Release_Date'] = pd.to_datetime(
    df['Release_Date'],
    format="%b %d, %Y",
    errors='coerce'
)

In [17]:
# Drop NaT Release_Date. These Rows are not needed
df = df.dropna(subset=['Release_Date'])

In [19]:
# Change Data types of 'GDP','GDI','GDP_%_Change','GDI_%_Change'
cols = ['GDP','GDI','GDP_%_Change','GDI_%_Change']
df[cols] = (
    df[cols]
    .replace(".....", pd.NA) # handle missing marker
    .replace(",", "", regex=True)     # remove thousands separators
    .apply(pd.to_numeric, errors="coerce")
)

In [21]:
# Create Quarter_Date column from Quarter.  
df['Quarter_Date'] = pd.PeriodIndex(df['Quarter'], freq='Q').to_timestamp(how='end').normalize()

In [24]:
# Reset Index
df = df.reset_index(drop=True)

In [ ]:
# Basic statistics
df.describe()

,GDP,GDI,GDP_%_Change,GDI_%_Change,Release_Date,Quarter_Date
count,1045.000000,928.000000,1045.000000,911.000000,1045,1045
mean,16644.757416,16512.985453,2.217990,2.128869,2016-03-31 09:20:50.526315776,2012-02-17 16:25:15.789473792
min,10313.100000,10386.900000,-32.900000,-33.500000,2002-04-26 00:00:00,2002-03-31 00:00:00
25%,13383.300000,13480.325000,1.300000,0.700000,2011-07-29 00:00:00,2006-06-30 00:00:00
50%,15242.900000,15111.900000,2.500000,2.300000,2017-07-28 00:00:00,2011-06-30 00:00:00
75%,19153.900000,18992.400000,3.500000,3.900000,2021-07-29 00:00:00,2017-03-31 00:00:00
max,31490.100000,30707.200000,35.300000,29.000000,2026-02-20 00:00:00,2025-12-31 00:00:00
std,4638.430269,4491.630231,4.850692,4.887844,NaN,NaN


In [31]:
# Latest Release_Date for each Quarter
df['Last_Release_Date'] = (
    df.groupby('Quarter')['Release_Date'].transform('max')
)

In [ ]:
# Create Valid_From and Valid_To
# First check if df is sorted via 'Quarter_Date', 'Release_Date'
# If is_sorted is True then Valid_From and Valid_To can be created
is_sorted = (
    df.groupby('Quarter_Date')['Release_Date']
    .apply(lambda s: s.is_monotonic_decreasing)
    .all()
)
is_sorted	   

True

In [ ]:
# Extra Check for possible quaters that are not sorted if is_sorted is False 
bad_quarters = (
    df.groupby("Quarter")["Release_Date"]
      .apply(lambda s: not s.is_monotonic_decreasing)
)
for item in bad_quarters:
    print(item)

False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False
False


In [42]:
# Create Valid_From and Valid_To
df['Valid_From'] = df['Release_Date']
df['Valid_To'] = (df.groupby('Quarter')['Release_Date'].shift(+1))

In [46]:
# Verify Validity Intervals Per Quarter
test = (
    df.sort_values(['Quarter_Date', 'Release_Date'])
      .groupby('Quarter')
      .apply(lambda x: (x['Valid_From'].shift(-1) < x['Valid_To']).any())
)
print(test[test])

Series([], dtype: bool)


In [47]:
# Build time series using validity intervals
# Use row where as_of >= Valid_From and (as_of <= Valid_To OR Valid_To is NaT)
def build_vintage_timeseries(df):
    value_cols = [
        'GDP',
        'GDI',
        'GDP_%_Change',
        'GDI_%_Change'
    ]

    # List of Unique Release_Date
    release_dates = (
        df['Release_Date']
        .dropna()
        .sort_values()
        .unique()
    )

    results ={}

    # Filling Results
    for as_of in release_dates:
        snapsh = df[
            (df['Valid_From'] <= as_of) &
            (
                (df['Valid_To'] > as_of) |
                df['Valid_To'].isna()
                  )
        ]

        snapsh = (
            snapsh[
                ['Quarter', 'Quarter_Date'] + value_cols
            ]
            .sort_values('Quarter_Date')
            .reset_index(drop=True)
        )

        results[pd.Timestamp(as_of)] = snapsh

    return results

In [48]:
# The time series are transfered as data frames into a dictionary
vint = build_vintage_timeseries(df)

In [49]:
type(vint)

dict

In [50]:
vint.keys()

dict_keys([Timestamp('2002-04-26 00:00:00'), Timestamp('2002-05-24 00:00:00'), Timestamp('2002-06-27 00:00:00'), Timestamp('2002-07-31 00:00:00'), Timestamp('2002-08-29 00:00:00'), Timestamp('2002-09-27 00:00:00'), Timestamp('2002-10-31 00:00:00'), Timestamp('2002-11-26 00:00:00'), Timestamp('2002-12-20 00:00:00'), Timestamp('2003-01-30 00:00:00'), Timestamp('2003-02-28 00:00:00'), Timestamp('2003-03-27 00:00:00'), Timestamp('2003-04-25 00:00:00'), Timestamp('2003-05-29 00:00:00'), Timestamp('2003-06-26 00:00:00'), Timestamp('2003-07-31 00:00:00'), Timestamp('2003-08-28 00:00:00'), Timestamp('2003-09-26 00:00:00'), Timestamp('2003-10-30 00:00:00'), Timestamp('2003-11-25 00:00:00'), Timestamp('2003-12-23 00:00:00'), Timestamp('2004-01-30 00:00:00'), Timestamp('2004-02-27 00:00:00'), Timestamp('2004-03-25 00:00:00'), Timestamp('2004-04-29 00:00:00'), Timestamp('2004-05-27 00:00:00'), Timestamp('2004-06-25 00:00:00'), Timestamp('2004-07-30 00:00:00'), Timestamp('2004-08-27 00:00:00'), Tim

In [51]:
# Alternatively transfer of the time series into single data frame 
panel = (
    pd.concat(vint, names=["As_Of_Release_Date"])
      .reset_index(level=0)
)

In [52]:
panel.head(10)

,As_Of_Release_Date,Quarter,Quarter_Date,GDP,GDI,GDP_%_Change,GDI_%_Change
0,2002-04-26,2002Q1,2002-03-31,10431.3,NaN,5.8,NaN
0,2002-05-24,2002Q1,2002-03-31,10428.8,10615.0,5.6,NaN
0,2002-06-27,2002Q1,2002-03-31,10449.8,10621.1,6.1,NaN
0,2002-07-31,2002Q1,2002-03-31,10313.1,10431.1,5.0,NaN
1,2002-07-31,2002Q2,2002-06-30,10369.9,NaN,1.1,NaN
0,2002-08-29,2002Q1,2002-03-31,10313.1,10431.1,5.0,NaN
1,2002-08-29,2002Q2,2002-06-30,10371.0,10536.7,1.1,NaN
0,2002-09-27,2002Q1,2002-03-31,10313.1,10431.1,5.0,NaN
1,2002-09-27,2002Q2,2002-06-30,10376.9,10541.5,1.3,NaN
0,2002-10-31,2002Q1,2002-03-31,10313.1,10431.1,5.0,NaN


In [29]:
df[['Quarter_Date', 'GDP']].loc[df['Release_Date'] == '2023-09-28']

,Quarter_Date,GDP
46,2023-06-30,27063.0
51,2023-03-31,26813.6
58,2022-12-31,26408.4
65,2022-09-30,25994.6
72,2022-06-30,25544.3
...,...,...
973,2003-03-31,11174.1
989,2002-12-31,11061.4
1003,2002-09-30,10984.0
1017,2002-06-30,10887.5


In [43]:
df.dtypes

Quarter                      object
Type                         object
GDP                         float64
GDI                         float64
GDP_%_Change                float64
GDI_%_Change                float64
Release_Date         datetime64[ns]
Note                         object
Quarter_Date         datetime64[ns]
Last_Release_Date    datetime64[ns]
Valid_From           datetime64[ns]
Valid_To             datetime64[ns]
dtype: object

In [44]:
df.head(20)

,Quarter,Type,GDP,GDI,GDP_%_Change,GDI_%_Change,Release_Date,Note,Quarter_Date,Last_Release_Date,Valid_From,Valid_To
0,2025Q4,Advance,31490.1,NaN,1.4,NaN,2026-02-20,GDI not published,2025-12-31,2026-02-20,2026-02-20,NaT
1,2025Q3,Updated,31098.0,30707.2,4.4,2.4,2026-01-22,<NA>,2025-09-30,2026-01-22,2026-01-22,NaT
2,2025Q3,Initial,31095.1,30706.5,4.3,2.4,2025-12-23,<NA>,2025-09-30,2026-01-22,2025-12-23,2026-01-22
3,2025Q2,Revised,30485.7,30244.9,3.8,2.6,2025-12-23,GDP not open for revision,2025-06-30,2025-12-23,2025-12-23,NaT
4,2025Q2,Third,30485.7,30335.8,3.8,3.8,2025-09-25,<NA>,2025-06-30,2025-12-23,2025-09-25,2025-12-23
5,2025Q2,Second,30353.9,30384.1,3.3,4.8,2025-08-28,<NA>,2025-06-30,2025-12-23,2025-08-28,2025-09-25
6,2025Q2,Advance,30331.1,NaN,3.0,NaN,2025-07-30,GDI not published,2025-06-30,2025-12-23,2025-07-30,2025-08-28
7,2025Q1,Revised,30042.1,29895.7,-0.6,1.0,2025-09-25,<NA>,2025-03-31,2025-09-25,2025-09-25,NaT
8,2025Q1,Third,29962.0,29885.7,-0.5,0.2,2025-06-26,<NA>,2025-03-31,2025-09-25,2025-06-26,2025-09-25
9,2025Q1,Second,29976.6,29850.3,-0.2,-0.2,2025-05-29,<NA>,2025-03-31,2025-09-25,2025-05-29,2025-06-26
